In [2]:
!pip install transformers peft datasets accelerate bitsandbytes -q

In [3]:
!pip install -U torchao -q

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# ===== small custom dataset =====
data = {
    "text": [
        "Question: What is AI Engineering? Answer: AI Engineering is building applications powered by AI models like LLMs.",
        "Question: What is RAG? Answer: RAG means Retrieval Augmented Generation, combining search with language models.",
        "Question: What is fine-tuning? Answer: Fine-tuning means further training a model on specific data.",
    ] * 20
}
dataset = Dataset.from_dict(data)

# ===== Tokenize Function (labels added) =====
def tokenize(examples):
    result = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize, batched=True)

# ===== LoRA Config =====
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ===== Training =====
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=5,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


Step,Training Loss
5,8.323010
10,8.276879
15,8.193430
20,8.184812


TrainOutput(global_step=24, training_loss=8.241893768310547, metrics={'train_runtime': 2.5586, 'train_samples_per_second': 70.351, 'train_steps_per_second': 9.38, 'total_flos': 2949780602880.0, 'train_loss': 8.241893768310547, 'epoch': 3.0})